In [ ]:
from IPython.display import clear_output

%pip install catboost

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

In [ ]:
import kagglehub
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Simply Join the Path with the appropiate csv name
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:

# Here we read the first 5 rows using the head() function

df.head()

In [ ]:
# Task 3: Write your code here:

# here we display General information about data

df.info()

In [ ]:
# Task 4: Write your code here:

# And here we show the Statistical Information of the data

df.describe()

In [ ]:
# Task 1: Write your code here:

# General information
print("Dataset Information:")
df.info()

# Missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

# Total missing values in dataset
print("\nTotal missing values in dataset:")
print(df.isna().sum().sum())

# Percentage of missing values per column
print("\nPercentage of missing values per column:")
print(df.isna().mean() * 100)


In [ ]:
# So as we can see from the previous cell, there are a lot of Missing values across the dataset
# And since all features are numerica/continous we may use the Median to fill missing values (We won't use the mean since it's sensitive to Outliers!)

features = df.columns.tolist()

for feature in features:
    df[feature] = df[feature].fillna(df[feature].median())

# Now we check the number of missing values

print(f"Number of MIssing Values After Cleaning: {df.isna().sum().sum()}")

In [ ]:
# Task 2: Write your code here:

# Here we will check the number of duplicates in the dataset
print(f"Number of Duplicates in the Dataset: {df.duplicated().sum()}")

# Looks Like there's No Duplicates!! ALL GOOD ! :D

In [ ]:
# Task 3: Write your code here:

categ_cols = df.select_dtypes(include=['object']).columns

print(f"Categorical Columns we have in the Dataset: {categ_cols}")

# The List is Empty!! Therefore there are no Categorical Columns!

In [ ]:
# Task 4: Write your code here:


# Now we simply extract the feature names and then Drop the Target Column !! WE DONT TRANSFORM THE TARGET COLUMN!
# then we apply a standard scaler to the data
columns = df.columns.drop("Target").tolist()

scaler = StandardScaler()

df[columns] = scaler.fit_transform(df[columns])

# We Just check the Values now
df.head()

In [ ]:
# Task 5: Write your code here:

# Class distribution
target = df["Target"].value_counts()

print("Class distribution (counts):")
print(target)

print("\nClass distribution (percentages):")
print(target / target.sum() * 100)

# Visualization
plt.figure(figsize=(8, 4))
plt.bar(target.index.astype(str), target.values, edgecolor='Black')
plt.title("Class Distribution of targetendary")
plt.xlabel("Class Label")
plt.ylabel("Count")
plt.show()

# There's a clear class imbalance ratio!!
# As we can see Labels of Class 0 are: 14732
# While Labels of Class 1 are: 5269

# With a Clear Class Imbalance Ratio of:
#0    73.6%
#1    26.4%

# Take a look into the plot too!

In [ ]:
# Task 1: Write your code here:

X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)


accuracy_scores = []
f1_scores = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{5}")
    X_fold_train, X_fold_val = X.iloc[train_index], X.iloc[test_index]
    y_fold_train, y_fold_val = y.iloc[train_index], y.iloc[test_index]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    accuracy_scores.append(accuracy_score(y_fold_val, y_fold_pred))
    f1_scores.append(np.sqrt(f1_score(y_fold_val, y_fold_pred)))

accuracy_scores = np.array(accuracy_scores)
f1_scores = np.array(f1_scores)

print(f"5-Fold CV Results:")
print(f"Accuracy:  {accuracy_scores.mean():,.2f}%")
print(f"F1-Score: {f1_scores.mean():,.2f}%")

In [ ]:
# Task 1: Write your code here:

# Task 1: Write your code here:


# Here we first extract the Feature Cols (DO NOT INCLUDE THE TARGET!)
feature_cols = df.columns.drop("Target").tolist()

# Then Group the Feature Cols with it's importance from the model respectivly and plt the Feature importance plot
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], edgecolor='black')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

# As we can see here we plotted all Feature Importances!, WE CANT LOOK AT IT! SO LETS JUST GET THE TOP 5 HERE

In [ ]:
feature_importance = feature_importance.head()

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], edgecolor='black')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

# Now it Looks Good!, And the Golden Feature is the P_2 !!!!

In [ ]:
# Task 2: Write your code here:

# Since Our feature_importance dataframe has the features ordered based on the importance, we could just select the first index !, because it contains the feature with most importance
feature_importance.iloc[0]

In [ ]:
# Task Bonus: Write your code here:

X = df['P_2']
y = df['Target']

# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

accuracy_scores = []
f1_scores = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{5}")
    X_fold_train, X_fold_val = X.iloc[train_index], X.iloc[test_index]
    y_fold_train, y_fold_val = y.iloc[train_index], y.iloc[test_index]

    print(X_fold_train.shape)
    print(X_fold_val.shape)
    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)
    print(y_fold_pred.shape)
    print(y_fold_val.shape)

    # Calculate metrics
    accuracy_scores.append(accuracy_score(y_fold_val, y_fold_pred))
    f1_scores.append(np.sqrt(f1_score(y_fold_val, y_fold_pred)))

accuracy_scores = np.array(accuracy_scores)
f1_scores = np.array(f1_scores)

print(f"5-Fold CV Results:")
print(f"Accuracy:  {accuracy_scores.mean():,.2f}%")
print(f"F1-Score: {f1_scores.mean():,.2f}%")


# After printing Out the Shapes and trying to debug, i don't know what is the y_fold_pred is 0
